In [50]:
retail_df = spark.read.format("csv")\
                    .option("header", True)\
                    .option("inferSchema", True)\
                    .load(path = "/workspaces/pyspark_udemy_codespace/data/invoices.csv")
retail_df.show()
retail_df.printSchema()

+---------+---------+--------------------+--------+---------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+---------------+---------+----------+--------------+
|   536365|     NULL|WHITE HANGING HEA...|       6|01-12-2010 8.26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|01-12-2010 8.26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|01-12-2010 8.26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|01-12-2010 8.26|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|01-12-2010 8.26|     3.39|     17850|United Kingdom|
|   536365|    22752|SET 7 BABUSHKA NE...|       2|01-12-2010 8.26|     7.65|     17850|United Kingdom|
|   536365|    21730|GLASS STAR FROSTE...|       6|01-12-2010 8.

In [53]:
"""
Calculate total_value(quantity * price) for each invoice line item
"""
from pyspark.sql.functions import col, round

retail_df.withColumn("TotalPrice", round(col("UnitPrice") * col("Quantity"), 2)).show()

+---------+---------+--------------------+--------+---------------+---------+----------+--------------+----------+
|InvoiceNo|StockCode|         Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|TotalPrice|
+---------+---------+--------------------+--------+---------------+---------+----------+--------------+----------+
|   536365|     NULL|WHITE HANGING HEA...|       6|01-12-2010 8.26|     2.55|     17850|United Kingdom|      15.3|
|   536365|    71053| WHITE METAL LANTERN|       6|01-12-2010 8.26|     3.39|     17850|United Kingdom|     20.34|
|   536365|   84406B|CREAM CUPID HEART...|       8|01-12-2010 8.26|     2.75|     17850|United Kingdom|      22.0|
|   536365|   84029G|KNITTED UNION FLA...|       6|01-12-2010 8.26|     3.39|     17850|United Kingdom|     20.34|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|01-12-2010 8.26|     3.39|     17850|United Kingdom|     20.34|
|   536365|    22752|SET 7 BABUSHKA NE...|       2|01-12-2010 8.26|     7.65|   

In [57]:
# Using expression variable
total_value_expr = round(col("UnitPrice") * col("Quantity"), 2) # logic for column written here

retail_df.withColumn("TotalPrice", total_value_expr).show() # logic substituted using variable for the logic

+---------+---------+--------------------+--------+---------------+---------+----------+--------------+----------+
|InvoiceNo|StockCode|         Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|TotalPrice|
+---------+---------+--------------------+--------+---------------+---------+----------+--------------+----------+
|   536365|     NULL|WHITE HANGING HEA...|       6|01-12-2010 8.26|     2.55|     17850|United Kingdom|      15.3|
|   536365|    71053| WHITE METAL LANTERN|       6|01-12-2010 8.26|     3.39|     17850|United Kingdom|     20.34|
|   536365|   84406B|CREAM CUPID HEART...|       8|01-12-2010 8.26|     2.75|     17850|United Kingdom|      22.0|
|   536365|   84029G|KNITTED UNION FLA...|       6|01-12-2010 8.26|     3.39|     17850|United Kingdom|     20.34|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|01-12-2010 8.26|     3.39|     17850|United Kingdom|     20.34|
|   536365|    22752|SET 7 BABUSHKA NE...|       2|01-12-2010 8.26|     7.65|   

In [60]:
"""
Perform the following exploratory analysis on invoices data
    Can we make invoice numbers a numeric field?
    Analyize quantity to identify potentially invalid records
    Analyze unit price to identify potentially invalid records
"""

retail_df.describe(["InvoiceNo", "Quantity", "UnitPrice"]).show()
"""
+-------+-----------------+------------------+-----------------+
|summary|        InvoiceNo|          Quantity|        UnitPrice|
+-------+-----------------+------------------+-----------------+
|  count|           541909|            541909|           541909|
|   mean| 559965.752026781|  9.55224954743324| 4.61111362608295|
| stddev|13428.41728080562|218.08115785023466|96.75985306117951|
|    min|           536365|            -80995|        -11062.06|
|    max|          C581569|             80995|          38970.0|
+-------+-----------------+------------------+-----------------+
"""
# We cannot make the InvoiceNo into Numeric because MAX value in the field is a String

[Stage 154:>                                                        (0 + 2) / 2]

+-------+-----------------+------------------+-----------------+
|summary|        InvoiceNo|          Quantity|        UnitPrice|
+-------+-----------------+------------------+-----------------+
|  count|           541909|            541909|           541909|
|   mean| 559965.752026781|  9.55224954743324| 4.61111362608295|
| stddev|13428.41728080562|218.08115785023466|96.75985306117951|
|    min|           536365|            -80995|        -11062.06|
|    max|          C581569|             80995|          38970.0|
+-------+-----------------+------------------+-----------------+



'\n+-------+-----------------+------------------+-----------------+\n|summary|        InvoiceNo|          Quantity|        UnitPrice|\n+-------+-----------------+------------------+-----------------+\n|  count|           541909|            541909|           541909|\n|   mean| 559965.752026781|  9.55224954743324| 4.61111362608295|\n| stddev|13428.41728080562|218.08115785023466|96.75985306117951|\n|    min|           536365|            -80995|        -11062.06|\n|    max|          C581569|             80995|          38970.0|\n+-------+-----------------+------------------+-----------------+\n'

In [66]:
retail_df.summary().select('summary', 'InvoiceNo', 'Quantity', 'UnitPrice').show()
"""
+-------+-----------------+------------------+-----------------+
|summary|        InvoiceNo|          Quantity|        UnitPrice|
+-------+-----------------+------------------+-----------------+
|  count|           541909|            541909|           541909|
|   mean| 559965.752026781|  9.55224954743324| 4.61111362608295|
| stddev|13428.41728080562|218.08115785023466|96.75985306117951|
|    min|           536365|            -80995|        -11062.06|
|    25%|         547903.0|                 1|             1.25|
|    50%|         560686.0|                 3|             2.08|
|    75%|         571842.0|                10|             4.13|
|    max|          C581569|             80995|          38970.0|
+-------+-----------------+------------------+-----------------+
"""

[Stage 169:============================>                            (1 + 1) / 2]

+-------+-----------------+------------------+-----------------+
|summary|        InvoiceNo|          Quantity|        UnitPrice|
+-------+-----------------+------------------+-----------------+
|  count|           541909|            541909|           541909|
|   mean| 559965.752026781|  9.55224954743324| 4.61111362608295|
| stddev|13428.41728080562|218.08115785023466|96.75985306117951|
|    min|           536365|            -80995|        -11062.06|
|    25%|         547903.0|                 1|             1.25|
|    50%|         560686.0|                 3|             2.08|
|    75%|         571842.0|                10|             4.13|
|    max|          C581569|             80995|          38970.0|
+-------+-----------------+------------------+-----------------+



'\n+-------+-----------------+------------------+-----------------+\n|summary|        InvoiceNo|          Quantity|        UnitPrice|\n+-------+-----------------+------------------+-----------------+\n|  count|           541909|            541909|           541909|\n|   mean| 559965.752026781|  9.55224954743324| 4.61111362608295|\n| stddev|13428.41728080562|218.08115785023466|96.75985306117951|\n|    min|           536365|            -80995|        -11062.06|\n|    25%|         547903.0|                 1|             1.25|\n|    50%|         560686.0|                 3|             2.08|\n|    75%|         571842.0|                10|             4.13|\n|    max|          C581569|             80995|          38970.0|\n+-------+-----------------+------------------+-----------------+\n'

In [69]:
from pyspark.sql.functions import min, max, percentile

retail_df.select(min(col("UnitPrice")).alias("min_price"), 
                 max(col("UnitPrice")).alias("max_price"), 
                 percentile(col("UnitPrice"), 0.99).alias("99_percentile_price")).show()

+---------+---------+-------------------+
|min_price|max_price|99_percentile_price|
+---------+---------+-------------------+
|-11062.06|  38970.0|               18.0|
+---------+---------+-------------------+

